In [ ]:
import torch
import timm
import argparse
import os
from torchvision import transforms
from torch.utils.data import DataLoader
from PIL import Image

# ARGUMENTS

parser = argparse.ArgumentParser()
parser.add_argument('--model', type=str, required=True,
                    help='ResNet50 | EfficientNet-B0 | ConvNeXt-Tiny')
parser.add_argument('--weights', type=str, required=True,
                    help='Path to .pth file')
parser.add_argument('--data', type=str, required=True,
                    help='Path to test data folder')
parser.add_argument('--batch_size', type=int, default=32)
args = parser.parse_args()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Transforms we do to image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Model Builder for our chosen models

def build_model(name, num_classes):
    if name == 'ResNet50':
        return timm.create_model('resnet50', pretrained=False, num_classes=num_classes)
    elif name == 'EfficientNet-B0':
        return timm.create_model('efficientnet_b0', pretrained=False, num_classes=num_classes)
    elif name == 'ConvNeXt-Tiny':
        return timm.create_model('convnext_tiny', pretrained=False, num_classes=num_classes)
    else:
        raise ValueError("Invalid model name")

# Data loader scaled to use any test data (new/unseen too)

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform):
        self.paths = []
        for root_dir, _, files in os.walk(root):
            for f in files:
                if f.lower().endswith(('jpg','png','jpeg')):
                    self.paths.append(os.path.join(root_dir, f))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, path

dataset = TestDataset(args.data, transform)
loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False)

# Loading the model

NUM_CLASSES = 30   # here we can change the number of classes

model = build_model(args.model, NUM_CLASSES)
model.load_state_dict(torch.load(args.weights, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# Final inferencing

print("\nRunning inference...\n")

with torch.no_grad():
    for imgs, paths in loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(1)

        for p, path in zip(preds, paths):
            print(f"{path} -> class_{p.item()}")

print("\nDone.")